# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [3]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [4]:
import requests

# download a pdf from a url.
# first get the pdf as an object
file_url = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
pdf = requests.get(file_url)


In [5]:
# now save the pdf in the same folder as this notebook
filename = "Business_in_AI_2025.pdf"
with open(filename, 'wb') as f:
    f.write(pdf.content)

In [6]:
import pypdf
from langchain_core.documents import Document

# use example code from langchain documentation to read the text of the pdf

# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]

file_path = f"./{filename}"
docs = load_pdf_pages(file_path)
#print(len(docs)) # number of pages in the document

# this is example code from the assignment notebook to combine the text contents of
# each page of the pdf into one string
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

#print(document_text)

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from utils.clients import get_client
from pydantic import BaseModel
import os
os.environ["LANGSMITH_TRACING"] = "false" 
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

# load a model using example code from the course
client = get_client()

# make a quick call
response = client.responses.create(
    model = MODEL,
    input = 'Hello world!'
)

print(response.output_text)

Hello! How can I assist you today?


In [ ]:
# create a pydantic object to define a structured llm output

from typing import Optional
from pydantic import BaseModel, ConfigDict, Field, computed_field

class StructuredOutput(BaseModel):
    #model_config = ConfigDict(extra="allow")
    author: str=Field()
    title: str=Field()
    relevance: str=Field(description="a statement, no longer than one paragraph, \
        that explains why is this article relevant for an AI professional in their professional development.")
    summary: str=Field(description="a concise and succinct summary no longer than 1000 tokens.")
    tone: str=Field(description="the tone used to produce the summary")

    # not really sure how to get the token values in the output, or set max tokens for output
    # obviously it should be doable but I dont have time to figure out rn

    #@computed_field
    #def input_tokens(self) -> int:
    #    return self.usage.input_tokens
    #def output_tokens(self) -> int:
    #    return self.usage.output_tokens
    
    #inputtokens: int=Field(description="number of input tokens")
    #outputtokens: int=Field(description="number of output tokens")



In [102]:
system_prompt = "You will reply in formal academic writing."

prompt = f"""
    Please extract the relevant information from this document.
    Here's the text of the document:
    <text>
    {document_text}
    </text>
    Thank you.
"""

In [ ]:
# start with a client, responses API, and we want to parse

response = client.responses.parse(
    model=MODEL,
    input=[ # the input is a list of dictionaries
        {"role": "system", "content": system_prompt}, # the system prompt
        {
            "role": "user", "content": prompt, # the actual prompt
        },
    ],
    text_format=StructuredOutput, # give it a class
)

output = response.output_parsed

In [104]:
output

StructuredOutput(author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', title='The GenAI Divide: State of AI in Business 2025', relevance='This article provides critical insights into the current state of AI implementation, focusing on significant challenges and patterns in generative AI use. For AI professionals, understanding these dynamics can inform strategies for success in deploying AI technologies effectively, enhancing organizational learning and adaptation.', summary="The report details preliminary findings from MIT's Project NANDA on AI implementation in businesses, revealing that despite substantial investments ($30-40 billion), 95% of organizations fail to achieve any measurable financial return from generative AI (GenAI) initiatives, coining the term 'GenAI Divide.' While tools like ChatGPT see high adoption, actual business transformation is limited, with structural changes only apparent in two out of eight major sectors. The primary barriers to success

In [105]:
from IPython.display import display, Markdown

# obtain output using the method output_parsed
display(Markdown(output.author))
display(Markdown(output.title))
display(Markdown(output.relevance))
display(Markdown(output.summary))
display(Markdown(output.tone))


Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

The GenAI Divide: State of AI in Business 2025

This article provides critical insights into the current state of AI implementation, focusing on significant challenges and patterns in generative AI use. For AI professionals, understanding these dynamics can inform strategies for success in deploying AI technologies effectively, enhancing organizational learning and adaptation.

The report details preliminary findings from MIT's Project NANDA on AI implementation in businesses, revealing that despite substantial investments ($30-40 billion), 95% of organizations fail to achieve any measurable financial return from generative AI (GenAI) initiatives, coining the term 'GenAI Divide.' While tools like ChatGPT see high adoption, actual business transformation is limited, with structural changes only apparent in two out of eight major sectors. The primary barriers to successful implementation include organizational learning gaps, lack of integration into existing workflows, and poor customization of AI tools for specific business processes. Successful organizations leverage adaptable systems that learn from user feedback, creating partnerships rather than internal builds, and prioritize tools that can blend smoothly into their operational frameworks. This report emphasizes the necessity for organizations to focus on meaningful adaptations and collaborations with tech vendors to generate real value from their AI investments, while also shedding light on evolving workforce dynamics influenced by AI adoption.

Formal and analytical.

In [81]:
response.model_dump()

{'id': 'resp_0fab095514d8b723006a41c5a867cc81a2851f3fe6a975b1c7',
 'created_at': 1782695336.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-4o-mini-2024-07-18',
 'object': 'response',
 'output': [{'id': 'msg_0fab095514d8b723006a41c5a9402481a2b26def686d7bc18e',
   'content': [{'annotations': [],
     'text': '{"author":"MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari","title":"The GenAI Divide: State of AI in Business 2025","relevance":"This article is pertinent for AI professionals as it highlights the challenges and opportunities within the current business landscape regarding AI implementation, emphasizing the importance of adaptability and user-centric design in AI tools—critical areas for professional development and effective application in organizational contexts.","summary":"The report investigates the \'GenAI Divide,\' which reveals a stark contrast in AI adoption and transformation across organiz

In [91]:
response.usage.input_tokens

10884

In [92]:
response.usage.output_tokens

260

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
# use deepeval summarizationmetric to evaluate the summary from the previous response

from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import GPTModel
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

# create a representation of the model?

# from course example code. using their gateway to openai
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
#MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

# define metrics for the test cases

# create the metric for summarization
metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Is the summary less than 1000 tokens?",
        "Is the content of the summary factual?",
        "Is there missing, relevant information from the summary?",
        "Is the summary concise and succinct?",
        "Is the tone of the summary correct?"
    ]
)
# clarity metric
clarity_metric = GEval(
    name="Clarity",
    model=model,
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that’s easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Check that the response is factual."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

# professionalism metric
professionalism_metric = GEval(
    name="Professionalism",
    criteria="Assess the level of professionalism and expertise conveyed in the response.",
    # NOTE: you can only provide either criteria or evaluation_steps, and not both
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing.",
        "Check that the output is written in formal academic writing style."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=model,
)

pii_leakage_metric = GEval(
    name="PII Leakage",
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts.",
        "Penalize outputs with PII which does not use placeholders or isn't anonymized."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=model,
)
    

In [ ]:
# create one big function to perform all the evaluation steps and then output the results as a dictionary

def CustomEvalResponse (prompt, actual_output, model,
    summarization_metric, clarity_metric, professionalism_metric, safety_metric):

    # 1. SUMMARIZATION
    # create a test case
    summarization_test_case = LLMTestCase(input=prompt, actual_output=actual_output)

    # perform the evaluation
    # To run metric as a standalone
    # metric.measure(test_case)
    # print(metric.score, metric.reason)
    #summarization_result = evaluate(test_cases=[summarization_test_case], metrics=[metric])
    # got a metric, now need to measure against a test case?
    metric.measure(summarization_test_case) 

    # 2. CLARITY
    # create a test case
    clarity_test_case = LLMTestCase(input=prompt, actual_output=actual_output)
    # create the metric?
    #clarity_result = evaluate(test_cases=[clarity_test_case], metrics=[clarity_metric])
    # evaluate metric against test case
    clarity_metric.measure(clarity_test_case)

    # 4. PROFESSIONALISM
    # create a test case
    professionalism_test_case = LLMTestCase(input=prompt, actual_output=actual_output)
    # create the metric?
    #professionalism_result = evaluate(test_cases=[professionalism_test_case], metrics=[professionalism_metric])
    # evaluate metric against test case
    professionalism_metric.measure(professionalism_test_case)


    # 4. SAFETY
    # create a test case
    pii_leakage_test_case = LLMTestCase(input=prompt, actual_output=actual_output)
    # create the metric?
    #pii_leakage_result = evaluate(test_cases=[pii_leakage_test_case], metrics=[pii_leakage_metric])
    # evaluate metric against test case
    pii_leakage_metric.measure(pii_leakage_test_case)

    summary_eval_scores = {
        "SummarizationScore" : metric.score,
        "SummarizationReason" : metric.reason,
        "ClarityScore" : clarity_metric.score,
        "ClarityReason" : clarity_metric.reason,
        "ProfessionalismScore" : professionalism_metric.score,
        "ProfessionalismReason" : professionalism_metric.reason,
        "PIILeakageScore" : pii_leakage_metric.score,
        "PIILeakageReason" : pii_leakage_metric.reason,
    }

    # optional print scores
    #display(Markdown(f'**Score**: {professionalism_metric.score}'))
    #display(Markdown(f'**Reason**: {professionalism_metric.reason}'))

    return summary_eval_scores



In [165]:
# call the function
summary_eval_scores = CustomEvalResponse (prompt, output.summary, model,
    summarization_metric=metric, 
    clarity_metric=clarity_metric,
    professionalism_metric=professionalism_metric, 
    safety_metric=pii_leakage_metric)

# takes about 18 sec to run

Output()

Output()

Output()

Output()

In [166]:
summary_eval_scores

{'SummarizationScore': 0.7272727272727273,
 'SummarizationReason': "The score is 0.73 because the summary includes contradicting information that undermines the accuracy of the original text, specifically regarding MIT's Project NANDA. Additionally, it presents extra information that was not mentioned in the original text, which further detracts from its overall fidelity. However, the summary likely captures some of the essential points, which is why it retains a score above 0.7.",
 'ClarityScore': 0.8731058590348988,
 'ClarityReason': "The response uses clear and direct language, effectively avoiding jargon while explaining the complex findings of MIT's Project NANDA. It presents the results and barriers to AI implementation in a logical manner, making it easy to follow. However, there are minor areas where further simplification could enhance clarity, such as breaking down the specific challenges faced by organizations into clearer categories.",
 'ProfessionalismScore': 0.91469651288

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [138]:
prompt

'\n    Please extract the relevant information from this document.\n    Here\'s the text of the document:\n    <text>\n    pg. 1 \n \n \nThe GenAI Divide  \nSTATE OF AI IN \nBUSINESS 2025 \n \n \n \n \n \n \nMIT NANDA \nAditya Challapally \nChris Pease \nRamesh Raskar \nPradyumna Chari \nJuly 2025 \n\npg. 2 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nNOTES \nPreliminary Findings from AI Implementation Research from Project NANDA \nReviewers: Pradyumna Chari, Project NANDA \nResearch Period: January – June 2025 \nMethodology: This report is based on a multi-method research design that includes \na systematic review of over 300 publicly disclosed AI initiatives, structured \ninterviews with representatives from 52 organizations, and survey responses from \n153 senior leaders collected across four major industry conferences. \n Disclaimer: The views expressed in this report are solely those of the authors and \nreviewers and do not reflect the positions of any affiliated employer

In [215]:
# create an improvement prompt, ask a FM to generate a new prompt

improvement_system_prompt = f"""
    You are a helpful assistant.
"""

improvement_prompt = f"""
    Based on the original prompt, summary, and evaluation results, 
    create a new prompt for a foundation model which enhances the summary.

    Here's the original prompt:
    <original_prompt>
    {prompt}
    </original_prompt>

    Here's the original summary:
    <original_summary>
    {output.summary}
    </original_summary>

    Here's the evaluation results as a dictionary:
    <eval_results>
    {summary_eval_scores}
    </eval_results>
"""


# create a pydantic object, structured output for the prompt

class NewPromptOutput(BaseModel):
    #model_config = ConfigDict(extra="allow")
    newprompt: str=Field(description="a new prompt for a FM, specifying that text will be provided")
    explanation: str=Field(description="an explanation for the new prompt")


In [216]:
# ask the model to make a new prompt
new_prompt_response = client.responses.parse(
    model=MODEL,
    input=[ # the input is a list of dictionaries
        {"role": "system", "content": improvement_system_prompt}, # the system prompt
        {
            "role": "user", "content": improvement_prompt, # the actual prompt
        },
    ],
    text_format=NewPromptOutput, # give it a class
)

new_prompt_output = new_prompt_response.output_parsed

In [217]:
display(Markdown(new_prompt_output.newprompt))

Please provide a detailed and accurate summary of the key findings from the report on AI implementation by MIT's Project NANDA, emphasizing the following aspects: 1) The concept of the GenAI Divide and its implications for organizations; 2) The specific barriers to successful implementation of AI technologies in businesses; 3) The differences between successful and unsuccessful organizations in their AI adoption strategies; 4) Insights into the evolving workforce dynamics resulting from AI integration. Ensure clarity and coherence in presenting this information, and avoid adding extraneous details not present in the original text.

In [218]:
display(Markdown(new_prompt_output.explanation))

This new prompt specifies the key themes to focus on while summarizing the report, including the GenAI Divide, barriers to implementation, successful strategies of organizations, and workforce dynamics. It aims to enhance the accuracy and fidelity of the summary, addressing the evaluation results' concerns regarding clarity and relevance.

In [219]:
# put together the new prompt
new_prompt = f"""
    {new_prompt_output.newprompt}

    Here's the text of the document:
    <text>
    {document_text}
    </text>
"""

In [220]:
response_2 = client.responses.parse(
    model=MODEL,
    input=[ # the input is a list of dictionaries
        {"role": "system", "content": system_prompt}, # the system prompt
        {
            "role": "user", "content": new_prompt, # the actual prompt
        },
    ],
    text_format=StructuredOutput, # give it a class
)

output_2 = response_2.output_parsed

In [221]:
display(Markdown(output_2.summary))

The report from MIT's Project NANDA identifies the 'GenAI Divide', where despite $30-40 billion invested in generative AI, 95% of organizations fail to realize returns, indicating a disparity between high adoption and low transformation. Key barriers to successful implementation include brittle workflows, poor contextual learning, and misalignment with existing operations. Successful organizations demonstrate a distinct approach, focusing on customized, adaptable systems that improve over time, in contrast to unsuccessful ones who generally pursue internal builds. Moreover, evolving workforce dynamics suggest the selective displacement of tasks, particularly in customer support and administrative functions, rather than broad layoffs. The rise of a 'shadow AI economy' highlights that users often prefer consumer-grade tools, signaling a demand for systems that can adapt, learn, and integrate seamlessly into workflows. Ultimately, organizations that prioritize deep customization, process integration, and partnerships with external vendors are likely to navigate these challenges more effectively, potentially realizing greater returns in back-office operations despite the initial focus on front-office functions.

In [222]:
# perform evaluation again
summary_eval_scores_2 = CustomEvalResponse (new_prompt, output_2.summary, model,
    summarization_metric=metric, 
    clarity_metric=clarity_metric,
    professionalism_metric=professionalism_metric, 
    safety_metric=pii_leakage_metric)
    


Output()

Output()

Output()

Output()

In [223]:
summary_eval_scores

{'SummarizationScore': 0.7272727272727273,
 'SummarizationReason': "The score is 0.73 because the summary includes contradicting information that undermines the accuracy of the original text, specifically regarding MIT's Project NANDA. Additionally, it presents extra information that was not mentioned in the original text, which further detracts from its overall fidelity. However, the summary likely captures some of the essential points, which is why it retains a score above 0.7.",
 'ClarityScore': 0.8731058590348988,
 'ClarityReason': "The response uses clear and direct language, effectively avoiding jargon while explaining the complex findings of MIT's Project NANDA. It presents the results and barriers to AI implementation in a logical manner, making it easy to follow. However, there are minor areas where further simplification could enhance clarity, such as breaking down the specific challenges faced by organizations into clearer categories.",
 'ProfessionalismScore': 0.91469651288

In [224]:
summary_eval_scores_2

{'SummarizationScore': 0.5454545454545454,
 'SummarizationReason': "The score is 0.55 because the summary contains contradicting information regarding the barriers to effective AI implementation and misrepresents the relationship between internal builds and success rates. Additionally, it introduces extra information about the adoption gap and 'shadow AI economy' that is not present in the original text, further undermining its accuracy.",
 'ClarityScore': 0.7864140608705135,
 'ClarityReason': "The response uses clear and direct language, effectively communicating complex ideas about the 'GenAI Divide' and its implications for organizations. However, it does include some jargon, such as 'shadow AI economy' and 'brittle workflows', which could benefit from brief explanations for accessibility. Overall, the information is presented in an organized manner, although there are minor areas where further simplification could enhance clarity.",
 'ProfessionalismScore': 0.9622459338205509,
 'Pr

### Commenting on the results

With evaluation and feedback, it seems like the new prompt results in a lower summarization score but similar scores for clarity, professionalism, and PII leakage. I tried using the new prompt to generate new responses and evaluate them multiple times. Reading the reasons, they look pretty similar to me though so maybe there's just some variance in how the scores are calculated. I think I could have gotten more improvement by creating a better improvement prompt, like specifying how to use the scores and reasons to generate a new prompt.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
